## Camada Gold
<p>Na camada gold deste projeto, consumimos os dados gerados na camada Silver, gerando tabelas analíticas voltadas ao negócio</p>
<p>É produzido um resumo mensal dos preços por município e produto, além de um apecto geral para todo o municipio, que integra preços e volumes. Neste contexto, são explorados:</p>
<ol>
<l>Agregações</l>
<l>Joins</l>
<l>Window functions</l>
<l>Persistência em tabelas Delta</l>
</ol>

In [0]:
# Imports
from delta.tables import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

##### Definição de constantes

In [0]:
CATALOGO      = "ANP_Combustiveis"

silverSchema = "02_silver"
goldSchema   = "03_gold"

tablesSilver = {
    "PRECOS_REVENDA": f"`{CATALOGO}`.`{silverSchema}`.`precos_revenda`",
    "VENDAS_MUNICIPIO": f"`{CATALOGO}`.`{silverSchema}`.`vendas_municipio`",
}

tablesGold = {
    "RESUMO_PRECOS": f"`{CATALOGO}`.`{goldSchema}`.`resumo_precos`",
    "MERCADO_MUNICIPAL": f"`{CATALOGO}`.`{goldSchema}`.`mercado_municipal`",
}

fuelCategories = [
    "GASOLINA",
    "ETANOL",
    "DIESEL",
    "DIESEL S10",
    "GNV",
]

##### Definição da função auxiliar para atualizar tabela Dellta

In [0]:
# Função auxiliar para carregar dados na tabela Delta
def mergeToDelta(df: DataFrame, tableName: str, condition: str):
    deltaTable = DeltaTable.forName(spark, tableName)

    (
        deltaTable.alias("target")
        .merge(
            df.alias("source"),
            condition,
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"[Merge-DeltaTable] Carga concluída em: {tableName}")

### Resumo mensal dos preços
##### Leitura dos dados da camada Silver

In [0]:
# Leitura dos dados de preços da camada Silver
silverPricesDf = (
    spark.table(tablesSilver["PRECOS_REVENDA"])
    .select(
        "codigo_ibge",
        "municipio",
        "uf",
        "id_revenda",
        "produto",
        "familia_combustivel",
        "data_coleta",
        "valor_venda",
    )
)

# Leitura dos dados de vendas da camada Silver
silverSalesDf = (
    spark.table(tablesSilver["VENDAS_MUNICIPIO"])
    .select(
        "ano_referencia",
        "codigo_ibge",
        "municipio",
        "uf",
        "familia_combustivel",
        "volume_vendido",
    )
)

pricesTot = silverPricesDf.count()
salesTot  = silverSalesDf.count()

print(f"Total de registros de preços disponiveies na Silver: {pricesTot:,}")
print(f"Total de registros de verndas disponíveis na Silver: {salesTot:,}")

##### Resumo mensal dos preços

In [0]:
pricesResumeDf = (
    silverPricesDf
    .withColumn("ano_referencia", F.year("data_coleta"))
    .withColumn("mes_referencia", F.month("data_coleta"))
    .groupBy(
        "ano_referencia",
        "mes_referencia",
        "codigo_ibge",
        "municipio",
        "uf",
        "produto",
        "familia_combustivel",
    )
    .agg(
        F.round(F.avg("valor_venda"), 3)
        .cast("decimal(10,3)")
        .alias("preco_medio"),

        F.expr("percentile_approx(valor_venda, 0.5, 10000)")
        .cast("decimal(10,3)")
        .alias("preco_mediano"),

        F.min("valor_venda")
        .cast("decimal(10,3)")
        .alias("preco_minimo"),

        F.max("valor_venda")
        .cast("decimal(10,3)")
        .alias("preco_maximo"),

        F.coalesce(
            F.stddev_samp("valor_venda"),
            F.lit(0.0),
        )
        .cast("double")
        .alias("desvio_padrao_preco"),

        (F.max("valor_venda") - F.min("valor_venda"))
        .cast("decimal(10,3)")
        .alias("amplitude_preco"),

        F.countDistinct("id_revenda")
        .cast("long")
        .alias("quantidade_postos"),

        F.count(F.lit(1))
        .cast("long")
        .alias("quantidade_observacoes"),
    )
    .withColumn("processado_gold_em", F.current_timestamp())
)

mergeToDelta(
    pricesResumeDf,
    tablesGold["RESUMO_PRECOS"],
    """
        target.ano_referencia = source.ano_referencia AND
        target.mes_referencia = source.mes_referencia AND
        target.codigo_ibge = source.codigo_ibge AND
        target.produto = source.produto
    """,
)

display(
    pricesResumeDf
    .orderBy(
        F.desc("ano_referencia"),
        F.desc("mes_referencia"),
        "uf",
        "municipio",
        "produto",
    )
    .limit(20)
)

### Contextos municipais
##### Agregação das tabelas
<p>Agregação das tabelas de preços e vendas por ano, codigo do municiio, estado e tipo de combustível</p>

In [0]:
anualPricesDf = (
    silverPricesDf
    .withColumn("ano_referencia", F.year("data_coleta"))
    .groupBy(
        "ano_referencia",
        "codigo_ibge",
        "municipio",
        "uf",
        "familia_combustivel",
    )
    .agg(
        F.round(F.avg("valor_venda"), 3)
        .cast("decimal(10,3)")
        .alias("preco_medio"),

        F.countDistinct("id_revenda")
        .cast("long")
        .alias("quantidade_postos"),
    )
)

anualSalesDf = (
    silverSalesDf
    .groupBy(
        "ano_referencia",
        "codigo_ibge",
        "municipio",
        "uf",
        "familia_combustivel",
    )
    .agg(
        F.sum("volume_vendido")
        .cast("double")
        .alias("volume_vendido"),
    )
)

##### Avaliação do cenário anual dos combustiveis por estado
<p>Join das tabelas já agregadas e cálculo das variações anuais com window function (lag)</p>

In [0]:
baseMarketDf = (
    anualPricesDf.alias("p")
    .join(
        anualSalesDf.alias("v"),
        (F.col("p.ano_referencia") == F.col("v.ano_referencia"))
        & (F.col("p.codigo_ibge") == F.col("v.codigo_ibge"))
        & (F.col("p.familia_combustivel") == F.col("v.familia_combustivel")),
        "inner",
    )
    .select(
        F.col("p.ano_referencia"),
        F.col("p.codigo_ibge"),
        F.col("p.municipio"),
        F.col("p.uf"),
        F.col("p.familia_combustivel"),
        F.col("p.preco_medio"),
        F.col("v.volume_vendido"),
        F.col("p.quantidade_postos"),
    )
)

window = (
    Window
    .partitionBy("codigo_ibge", "familia_combustivel")
    .orderBy("ano_referencia")
)

marketDf = (
    baseMarketDf
    .withColumn(
        "ano_anterior",
        F.lag("ano_referencia").over(window),
    )
    .withColumn(
        "preco_ano_anterior",
        F.lag("preco_medio").over(window),
    )
    .withColumn(
        "vendas_ano_anterior",
        F.lag("volume_vendido").over(window),
    )
)

municipalMarketDf = (
    marketDf
    .select(
        "ano_referencia",
        "codigo_ibge",
        "municipio",
        "uf",
        "familia_combustivel",
        "preco_medio",
        "volume_vendido",
        "quantidade_postos",

        F.when(
            F.col("preco_ano_anterior").isNull()
            | (F.col("ano_anterior") != F.col("ano_referencia") - 1)
            | (F.col("preco_ano_anterior") == 0),
            F.lit(None).cast("double"),
        )
        .otherwise(
            F.round(
                (
                    (F.col("preco_medio") - F.col("preco_ano_anterior"))
                    / F.col("preco_ano_anterior")
                ) * 100,
                2,
            ).cast("double")
        )
        .alias("variacao_anual_preco"),

        F.when(
            F.col("vendas_ano_anterior").isNull()
            | (F.col("ano_anterior") != F.col("ano_referencia") - 1)
            | (F.col("vendas_ano_anterior") == 0),
            F.lit(None).cast("double"),
        )
        .otherwise(
            F.round(
                (
                    (F.col("volume_vendido") - F.col("vendas_ano_anterior"))
                    / F.col("vendas_ano_anterior")
                ) * 100,
                2,
            ).cast("double")
        )
        .alias("variacao_anual_vendas"),

        F.current_timestamp().alias("processado_gold_em"),
    )
)

mergeToDelta(
    municipalMarketDf,
    tablesGold["MERCADO_MUNICIPAL"],
    """
        target.ano_referencia = source.ano_referencia AND
        target.codigo_ibge = source.codigo_ibge AND
        target.familia_combustivel = source.familia_combustivel
    """,
)

display(
    municipalMarketDf
    .orderBy(
        F.desc("ano_referencia"),
        "uf",
        "municipio",
        "familia_combustivel",
    )
    .limit(20)
)

##### Pivotação das categorias de combustiveis
<p>A pivotação permite identificar a variação dos combustiveis ao longo dos anos</p>

In [0]:
precosPivotDf = (
    silverPricesDf
    .withColumn("ano_referencia", F.year("data_coleta"))
    .groupBy("ano_referencia")
    .pivot("familia_combustivel", fuelCategories)
    .agg(F.round(F.avg("valor_venda"), 3))
    .orderBy("ano_referencia")
)

display(precosPivotDf)

##### Consulta com spark SQL
<p>Exibir maiores variações anuais dos preços.</p>

In [0]:
municipalMarketDf.createOrReplaceTempView("vw_mercado_municipal_gold")

maioresVariacoesDf = spark.sql(
    """
        SELECT
            ano_referencia,
            municipio,
            uf,
            familia_combustivel,
            preco_medio,
            volume_vendido,
            variacao_anual_preco,
            variacao_anual_vendas
        FROM vw_mercado_municipal_gold
        WHERE variacao_anual_preco IS NOT NULL
        ORDER BY ABS(variacao_anual_preco) DESC
        LIMIT 20
    """
)

display(maioresVariacoesDf)